# 02. Entrenamiento de Modelos EURUSD (Optimizado)

Este notebook consume los datos preprocesados (`df_final_features.parquet`) para entrenar modelos predictivos de manera eficiente.

**Novedades:**
- **Selección de Modelos:** RandomForest vs LightGBM.
- **Grid Search:** Búsqueda de hiperparámetros básica.
- **Modelos Independientes:** Configuración separada para Compra y Venta.
- **Evaluación Avanzada:** Análisis de umbrales con `analizar_metricas_por_umbral`.

In [2]:
import pandas as pd
import numpy as np
import os
import sys
import joblib
import traceback
import json
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import TruncatedSVD 
from sklearn.utils import class_weight
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from lightgbm import LGBMClassifier

# Ruta absoluta al directorio src para importar utils
src_path = os.path.abspath("../src")
if src_path not in sys.path:
    sys.path.append(src_path)

from utils.viz_utils import analizar_metricas_por_umbral

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 1. Cargar Datos Procesados

In [3]:
processed_data_dir = '../data/processed/EURUSD'
final_features_path = os.path.join(processed_data_dir, 'df_final_features.parquet')
model_dir = '../models/EURUSD'
clusterer_filepath = os.path.join(model_dir, 'forex_candle_clusterer_eurusd_5mins.joblib')

print("Loading processed data...")
try:
    df_final = pd.read_parquet(final_features_path)
    print(f"Loaded DataFrame shape: {df_final.shape}")
    
    if os.path.exists(clusterer_filepath):
        fitted_clusterer = joblib.load(clusterer_filepath)
        print("Clusterer loaded.")
    else:
        print("Warning: Clusterer file not found. Ensure it was generated in step 01 if needed later.")

except Exception as e:
    print(f"Error loading data: {e}")
    raise

Loading processed data...
Loaded DataFrame shape: (1105411, 632)
Clusterer loaded.


## 2. Definir Fechas de Corte y Split

In [4]:
split_date_train_end = "2023-12-31" 
split_date_test_end = "2024-12-31"

if df_final['datetime'].dt.tz is None:
    df_final['datetime'] = df_final['datetime'].dt.tz_localize('UTC')

tz_info = df_final['datetime'].dt.tz
split_datetime_train_end = pd.Timestamp(split_date_train_end, tz=tz_info).replace(hour=23, minute=59, second=59)
split_datetime_test_end = pd.Timestamp(split_date_test_end, tz=tz_info).replace(hour=23, minute=59, second=59)

df_train = df_final[df_final['datetime'] <= split_datetime_train_end].copy()
df_test = df_final[(df_final['datetime'] > split_datetime_train_end) & (df_final['datetime'] <= split_datetime_test_end)].copy()
df_oot = df_final[df_final['datetime'] > split_datetime_test_end].copy()

print(f"Train rows: {len(df_train)}")
print(f"Test rows:  {len(df_test)}")
print(f"OOT rows:   {len(df_oot)}")

Train rows: 1009598
Test rows:  72449
OOT rows:   23364


## 3. Preparación para Entrenamiento (X, y)

In [5]:
TARGET_VAR_BUY = 'target_buy'
TARGET_VAR_SELL = 'target_sell'

def prepare_training_data(df, target_col):
    valid_idx = df.dropna(subset=['pattern']).index
    df_clean = df.loc[valid_idx]
    
    y = df_clean[target_col]
    
    cols_to_drop = [
        'datetime', 'open', 'close', 'high', 'low', 'volume', 'Date',
        'barCount', 'Year', 'Minute', 
        TARGET_VAR_BUY, TARGET_VAR_SELL, 'multiclass_target',
        'compra_resultado', 'venta_resultado', 'compra_velas', 'venta_velas', 'compra_pips', 'venta_pips',
        'future_high_values', 'future_low_values', 'future_close_values'
    ]
    cols_to_drop += [c for c in df_clean.columns if c.startswith('future_') or c.startswith('target_')]
    
    X = df_clean.drop(columns=cols_to_drop, errors='ignore')
    return X, y

print("Preparing Training Data...")
X_train, y_train_buy = prepare_training_data(df_train, TARGET_VAR_BUY)
_, y_train_sell = prepare_training_data(df_train, TARGET_VAR_SELL)

# Preparar Test Data para evaluación posterior
X_test, y_test_buy = prepare_training_data(df_test, TARGET_VAR_BUY)
_, y_test_sell = prepare_training_data(df_test, TARGET_VAR_SELL)

# Guardar df_test limpio con resultados originales para analisis con viz_utils
df_test_eval = df_test.loc[X_test.index].copy()

print(f"X_train shape: {X_train.shape}")

Preparing Training Data...
X_train shape: (1009598, 613)


## 4. Configuración del Pipeline y Model Selection

In [6]:
# --- 1. Preprocesadores ---
potential_num = X_train.select_dtypes(include=np.number).columns.tolist()
other_cat = [c for c in ['DayOfWeek', 'Month', 'Day', 'Hour', 'candle_cluster'] if c in X_train.columns]
pattern_feat = ['pattern'] if 'pattern' in X_train.columns else []
final_num = [c for c in potential_num if c not in other_cat and c not in pattern_feat]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_ohe_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

N_EMBEDDING_COMPONENTS = 30
pattern_embedding_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True)),
    ('svd', TruncatedSVD(n_components=N_EMBEDDING_COMPONENTS, random_state=42))
])

transformers = [('num', numeric_transformer, final_num)]
if other_cat: transformers.append(('cat_ohe', categorical_ohe_transformer, other_cat))
if pattern_feat: transformers.append(('pattern_embed', pattern_embedding_transformer, pattern_feat))

preprocessor = ColumnTransformer(transformers, remainder='drop', verbose=False)

# --- 2. Definición de Modelos y Grids ---

# Opciones de Modelo
MODELS = {
    'RandomForest': RandomForestClassifier(random_state=42, n_jobs=-1, class_weight='balanced'),
    'LGBM': LGBMClassifier(random_state=42, n_jobs=-1, class_weight='balanced', verbose=-1)
}

# Grids para búsqueda de hiperparámetros (simplificados)
PARAM_GRIDS = {
    'RandomForest': {
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [10, 20],
        'classifier__min_samples_leaf': [20, 50]
    },
    'LGBM': {
        'classifier__n_estimators': [100, 200],
        'classifier__learning_rate': [0.01, 0.05],
        'classifier__num_leaves': [31, 50]
    }
}

# --- 3. Función de Entrenamiento con Selección y Calibración ---
def train_calibrated_model(X, y, model_name='RandomForest', perform_grid_search=True):
    print(f"Training {model_name}...")
    base_model = MODELS[model_name]
    
    # Crear Pipeline Base (sin calibración aún, para el grid search)
    pipeline = Pipeline(steps=[('preprocessing', preprocessor), ('classifier', base_model)])
    
    best_estimator = pipeline
    
    if perform_grid_search:
        print(f"   Running Grid Search for {model_name}...")
        param_grid = PARAM_GRIDS[model_name]
        # Usar TimeSeriesSplit para evitar data leakage en validación cruzada
        cv = TimeSeriesSplit(n_splits=3)
        
        grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring='f1', n_jobs=1, verbose=1)
        grid_search.fit(X, y)
        
        print(f"   Best Params: {grid_search.best_params_}")
        best_estimator = grid_search.best_estimator_
    else:
        best_estimator.fit(X, y)

    # --- Calibración ---
    print("   Calibrating model (CV method)...")
    # Extraemos parámetros del mejor estimador para crear una nueva instancia limpia
    best_clf_params = best_estimator.named_steps['classifier'].get_params()
    new_base_clf = MODELS[model_name].set_params(**best_clf_params)
    
    # Calibración Cross-Validated (re-entrena internamente)
    calibrated_clf = CalibratedClassifierCV(new_base_clf, method='isotonic', cv=3)
    
    # Pipeline final calibrado
    final_pipeline = Pipeline(steps=[('preprocessing', preprocessor), ('classifier', calibrated_clf)])
    final_pipeline.fit(X, y)
    
    return final_pipeline

## 5. Ejecución del Entrenamiento (Configurable)

In [ ]:
# Configuración de Modelos: Elige 'RandomForest' o 'LGBM'
# Puedes usar modelos diferentes para Compra y Venta
BUY_MODEL_TYPE = 'LGBM' 
SELL_MODEL_TYPE = 'LGBM'
DO_GRID_SEARCH = True

print(f"Training BUY Model using {BUY_MODEL_TYPE}...")
pipeline_buy = train_calibrated_model(X_train, y_train_buy, model_name=BUY_MODEL_TYPE, perform_grid_search=DO_GRID_SEARCH)
joblib.dump(pipeline_buy, os.path.join(model_dir, f'forex_model_eurusd_5mins_BUY_{BUY_MODEL_TYPE}.joblib'))
print("BUY Model trained and saved.")

print(f"\nTraining SELL Model using {SELL_MODEL_TYPE}...")
pipeline_sell = train_calibrated_model(X_train, y_train_sell, model_name=SELL_MODEL_TYPE, perform_grid_search=DO_GRID_SEARCH)
joblib.dump(pipeline_sell, os.path.join(model_dir, f'forex_model_eurusd_5mins_SELL_{SELL_MODEL_TYPE}.joblib'))
print("SELL Model trained and saved.")

## 6. Evaluación Avanzada
Evaluamos usando `analizar_metricas_por_umbral` para encontrar las mejores zonas operativas. 
Los modelos calibrados tienden a dar probabilidades más "honestas" (más bajas), por lo que los umbrales óptimos suelen ser menores a 0.5.

In [ ]:
print("Evaluating Models on TEST Set...")

# Predecir probabilidades (la clase positiva es 1)
prob_buy = pipeline_buy.predict_proba(X_test)
prob_sell = pipeline_sell.predict_proba(X_test)

# Construir matriz combinada para la visualización
# Shape requerida: (n_samples, 3) -> [prob_neutral, prob_compra, prob_venta]
y_pred_proba_combined = np.zeros((len(X_test), 3))
y_pred_proba_combined[:, 1] = prob_buy[:, 1]  # Columna 1: Probabilidad Compra
y_pred_proba_combined[:, 2] = prob_sell[:, 1] # Columna 2: Probabilidad Venta
y_pred_proba_combined[:, 0] = 1 - y_pred_proba_combined[:, 1] - y_pred_proba_combined[:, 2] # Resto

# Ejecutar Análisis Visual
compra_res, venta_res, compra_m, venta_m = analizar_metricas_por_umbral(
    y_pred_proba_combined, 
    df_test_eval, 
    pip_objetivo=25,  # Pips configurados en targets
    pip_stop=10,      # Stop Loss configurado
    pip_sin_alcance=-1 # Pequeña penalización por capital estancado
)